# 09 – Person Voice Works

Esplorazione e data cleaning del dataset `person_voice_works.csv`.

| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona (doppiatore/attrice) su MyAnimeList |
| `role` | Tipo di ruolo |
| `anime_mal_id` | ID univoco dell'anime su MAL |
| `character_mal_id` | ID univoco del personaggio doppiato su MAL |
| `language` | Lingua del doppiaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_pvw = pd.read_csv('../datasets/person_voice_works.csv')
print(f'Shape: {df_pvw.shape}')
df_pvw.info()
df_pvw.head()

**Osservazioni iniziali:**
- Il dataset contiene 489,516 righe e tutte le colonne sono popolate e non contengono valori nulli.
- I tipi di dati sono adeguati.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [ ]:
n_originale = len(df_pvw)

mask_dup = df_pvw.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pvw[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pvw.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pvw):,}')

**Osservazioni:**

**1,488 righe** risultano coinvolte in duplicazioni esatte su tutte le colonne. Di queste, **537** sono prime occorrenze mantenute e **951** occorrenze extra rimosse. Dopo la rimozione il dataset scende da 489,516 a **488,565 righe**.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `person_details.csv`.

I valori duplicati sono **attesi**: lo stesso doppiatore può aver lavorato a più anime/personaggi in lingue diverse.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `person_details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [ ]:
df_persons = pd.read_csv('../datasets_cleaned/person_details_clean.csv')

mask_orphan_person = check_fk(df_pvw['person_mal_id'], df_persons['person_mal_id'], child_df=df_pvw)

print(f'Null in person_mal_id               : {df_pvw["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id (attesi) : {df_pvw["person_mal_id"].duplicated().sum():,}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID di riferimento valido.
- **Integrità referenziale**: non ci sono righe orfane.

**Nessuna pulizia necessaria.**

### 2.2 `role`

Colonna che indica il tipo di ruolo del doppiatore. Ci interessa verificare la presenza di valori nulli e anomalie. Per questo utilizziamo `analyze`. I duplicati sono **attesi**.

In [ ]:
df_pvw['role'] = df_pvw['role'].str.strip()
analyze(df_pvw['role'])

**Osservazioni:**
- Nessun null. Tutte le righe hanno un valore per il ruolo.
- Solo due valori distinti (`Main` e `Supporting`).

**Nessuna pulizia necessaria.**

### 2.3 `anime_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: ogni anime coinvolge più doppiatori per più personaggi.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [ ]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_anime = check_fk(df_pvw['anime_mal_id'], df_details['mal_id'], child_df=df_pvw)

print(f'Null in anime_mal_id               : {df_pvw["anime_mal_id"].isna().sum()}')
print(f'Duplicati in anime_mal_id (attesi) : {df_pvw["anime_mal_id"].duplicated().sum():,}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Integrità referenziale**: non ci sono righe orfane.

**Nessuna pulizia necessaria.**

### 2.4 `character_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `character_mal_id` di `characters.csv`.

I valori duplicati sono **attesi**: lo stesso personaggio può essere doppiato da più persone in lingue diverse.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `characters_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [ ]:
df_characters = pd.read_csv('../datasets_cleaned/characters_clean.csv')

mask_orphan_char = check_fk(df_pvw['character_mal_id'], df_characters['character_mal_id'], child_df=df_pvw)

print(f'Null in character_mal_id               : {df_pvw["character_mal_id"].isna().sum()}')
print(f'Duplicati in character_mal_id (attesi) : {df_pvw["character_mal_id"].duplicated().sum():,}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID personaggio valido.
- **Integrità referenziale**: non ci sono righe orfane.

**Nessuna pulizia necessaria.**